# Практика RG "Математические основы ML"

Для полноценной работы программы необходимо в одну папку с ноутбуком загрузить файлы `checker.py` и `cars.csv`

## Блок 1: Основные операции NumPy

### Векторы и матрицы

В машинном обучении важно различать:
- **1D-массив:** одномерный вектор размера `(n,)`. У него нет явного понятия строки или столбца.
- **2D-массив (матрица):** таблица размера `(N, d)`, где $N$ — число объектов (строк), $d$ — число признаков (столбцов).

Посмотрим на базовые способы создания и проверки размерностей:

In [ ]:
import numpy as np

# 1. Одномерный вектор (shape: (n,))
v = np.array([1.5, 2.0, 3.5])
print("Вектор v:", v)
print("Форма (shape) v:", v.shape)
print("Размерность (ndim) v:", v.ndim)
print(" ")

# 2. Двумерная матрица 3x2 (3 объекта, 2 признака)
X = np.array([
    [10.0, 20.0],
    [15.0, 25.0],
    [12.0, 22.0]
])
print("Матрица X:\n", X)
print("Размерности (N строк, d столбцов):", X.shape)
print(" ")

# 3. Случайные матрицы и векторы (понадобятся для инициализации весов)
# np.random.randn генерирует числа из стандартного нормального распределения
np.random.seed(42)
w_random = np.random.randn(2)
print("Случайный вектор весов w:", np.round(w_random, 3))

### Срезы и индексация

Срез матрицы в NumPy записывается как `матрица[строки, столбцы]`:
- Двоеточие `:` означает «взять все элементы вдоль этой оси».
- Срез вида `X[:, 0]` берет **все строки нулевого столбца** (например, конкретный признак).

In [ ]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12]
])

print("Вся матрица:\n", A)
print("Первый объект (строка с индексом 0):", A[0, :])
print("Второй признак (столбец с индексом 1):", A[:, 1])
print("Подматрица (первые 2 строки, первые 2 столбца):\n", A[:2, :2])
print(" ")

# Превращение 1D вектора в вектор-столбец (N, 1) через reshape:
vec_col = A[:, 1].reshape(-1, 1)
print("Столбец после reshape(-1, 1):\n", vec_col)
print("Его новая форма:", vec_col.shape)

### Агрегации вдоль осей: `axis=0` и `axis=1`

В анализе данных и ML это ключевой концепт:
- `axis=0` — схлопывание **по строкам** (двигаемся сверху вниз). Результат — характеристика **каждого признака** (например, среднее значение каждого признака по всем объектам).
- `axis=1` — схлопывание **по столбцам** (двигаемся слева направо). Результат — характеристика **каждого объекта** (например, сумма признаков одного пациента).
- Без указания оси — агрегация по **всему** массиву в одно число.

In [ ]:
data = np.array([
    [10.0, 100.0],
    [20.0, 200.0],
    [30.0, 300.0]
])

# Среднее по признакам (по колонкам) -> axis=0
feature_means = data.mean(axis=0)
print("Исходная матрица:\n", data)
print("Среднее каждого признака (axis=0):", feature_means)

# Сумма по объектам (по строкам) -> axis=1
row_sums = data.sum(axis=1)
print("Сумма характеристик каждого объекта (axis=1):", row_sums)

# Общее среднее по всей матрице
total_mean = data.mean()
print("Общее среднее всех элементов:", total_mean)

### Broadcasting (поэлементные операции без циклов)

**Broadcasting** — механизм NumPy, позволяющий выполнять арифметические операции над массивами разной формы без явных циклов `for`.

Классический пример в ML — **центрирование матрицы данных**:
Вычитание вектора средних значений признаков $\mu \in \mathbb{R}^d$ из каждого объекта выборки $X \in \mathbb{R}^{N \times d}$.

In [ ]:
X = np.array([
    [10.0, 100.0],
    [20.0, 200.0],
    [30.0, 300.0]
])

means = X.mean(axis=0)

# Массив формы (3, 2) минус массив формы (2,)
# NumPy автоматически "растянет" вектор means по всем строкам:
X_centered = X - means

print("Вектор средних:", means)
print("Центрированная матрица (X - means):\n", X_centered)
print("Проверка: среднее центрированной матрицы по колонкам:", X_centered.mean(axis=0))

### Линейная алгебра: матричное умножение (`@`), транспонирование и Bias Trick

Вспомним формулы из презентации:
1. **Прогноз для одного объекта:** $\hat{y} = w^T x = \langle w, x \rangle$. В Python для скалярного произведения векторов используется оператор `@` (или `np.dot`).
2. **Прогноз сразу для всей выборки:** $\hat{y} = X \tilde{w}$, где к матрице $X$ слева добавлен столбец единиц ($x_0 \equiv 1$).
3. **Транспонирование:** свойство `.T` меняет строки и столбцы местами.

In [ ]:
# 1. Скалярное произведение векторов
x = np.array([2.0, 3.0])
w = np.array([0.5, 1.5])
dot_product = x @ w  # 2.0*0.5 + 3.0*1.5 = 1.0 + 4.5 = 5.5
print(f"Скалярное произведение x @ w: {dot_product}")
print(" ")

# 2. Матричное умножение (прогноз для выборки из 3 объектов и 2 признаков)
X_features = np.array([
    [2.0, 3.0],
    [1.0, 4.0],
    [5.0, 0.0]
]) # shape: (3, 2)

# Добавление фиктивной единицы слева (Bias Trick: x_0 = 1)
# np.ones((N, 1)) создает столбец единиц, np.c_ склеивает столбцы
bias_col = np.ones((X_features.shape[0], 1))
X_with_bias = np.c_[bias_col, X_features]  # shape: (3, 3)

w_with_bias = np.array([10.0, 0.5, 1.5])

y_hat = X_with_bias @ w_with_bias

print("Матрица с добавленной единицей:\n", X_with_bias)
print("Вектор предсказаний y_hat (X @ w):", y_hat)
print(" ")

# 3. Транспонирование
print("Форма X_features:", X_features.shape)
print("Форма X_features.T:", X_features.T.shape)

#### Задание 1.1 Разделение матрицы признаков и вектора таргета

В табличных данных часто последний столбец матрицы представляет собой целевую переменную $y$, а все предыдущие — матрицу признаков $X$.

Реализуйте функцию `split_features_target(data)`:
- На вход подается 2D-массив `data` размерности $(N, d+1)$.
- Функция должна вернуть кортеж `(X, y)`:
  - `X` — 2D-массив всех столбцов, кроме последнего, размерности $(N, d)$.
  - `y` — 1D-массив последнего столбца размерности $(N,)$.

*Ограничение: использовать только базовые срезы NumPy (без копирования в циклах).*

In [ ]:
def split_features_target(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Разделяет матрицу data на признаки X (все столбцы, кроме последнего)
    и одномерный вектор целевой переменной y (последний столбец).
    """
    # YOUR CODE HERE
    pass

#### Задание 1.2. Выборочные статистики признаков

При анализе данных важно уметь считать статистики раздельно для каждого признака.

Реализуйте функцию `compute_column_stats(X)`:
- Принимает 2D-массив `X` размерности $(N, d)$.
- Возвращает кортеж `(means, stds)`:
  - `means` — 1D-массив выборочных средних каждого столбца размерности $(d,)$.
  - `stds` — 1D-массив стандартных отклонений каждого столбца размерности $(d,)$.

*Ограничение: обязательно используйте аргумент `axis` встроенных методов NumPy.*

In [5]:
def compute_column_stats(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Вычисляет среднее и стандартное отклонение для каждого столбца матрицы X."""
    # YOUR CODE HERE
    pass

#### Задание 1.3. Центрирование данных

Центрирование матрицы заключается в вычитании среднего значения каждого признака из всех строк:
$$\tilde{x}_{ij} = x_{ij} - \mu_j$$

Реализуйте функцию `center_matrix(X)`:
- Принимает 2D-массив `X` размерности $(N, d)$.
- Возвращает центрированную матрицу той же размерности $(N, d)$, в которой среднее по каждому столбцу равно $0$[cite: 20].

*Ограничение: используйте broadcasting. Запрещено использовать циклы `for` и `while`.*

In [6]:
def center_matrix(X: np.ndarray) -> np.ndarray:
    """Вычитает среднее каждого признака из матрицы X с помощью broadcasting."""
    # YOUR CODE HERE
    pass

#### Задание 1.4. Трюк с фиктивной единицей (Bias Trick)

Чтобы представить линейную модель со свободным коэффициентом $w_0$ в едином матричном виде $\hat{y} = X\tilde{w}$, к матрице $X$ слева добавляется столбец, состоящий строго из единиц ($x_0 \equiv 1$).

Реализуйте функцию `add_bias_term(X)`:
- Принимает 2D-массив `X` размерности $(N, d)$.
- Возвращает расширенную матрицу $\tilde{X}$ размерности $(N, d + 1)$, где первый столбец (с индексом 0) целиком состоит из единиц `1.0`.

In [7]:
def add_bias_term(X: np.ndarray) -> np.ndarray:
    """Добавляет столбец единиц слева к матрице признаков X."""
    # YOUR CODE HERE
    pass

#### Задание 1.5. Расчет предсказаний модели

Формула линейного предсказания для всей выборки имеет матричный вид:
$$\hat{y} = Xw$$

Реализуйте функцию `predict_linear(X, w)`:
- Принимает матрицу признаков `X` размерности $(N, d)$ и вектор весов `w` размерности $(d,)$.
- Возвращает одномерный вектор предсказаний $\hat{y}$ размерности $(N,)$.

*Ограничение: используйте матричный оператор `@` или `np.dot`.*

In [8]:
def predict_linear(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Вычисляет предсказания линейной модели y_pred = X @ w."""
    # YOUR CODE HERE
    pass

Протестируйте полученные функции

In [ ]:
from checker import (
    test_add_bias_term,
    test_center_matrix,
    test_compute_column_stats,
    test_predict_linear,
    test_split_features_target,
)

test_split_features_target(split_features_target)
test_compute_column_stats(compute_column_stats)
test_center_matrix(center_matrix)
test_add_bias_term(add_bias_term)
test_predict_linear(predict_linear)

## Блок 2. Описательные статистики, форма распределения и правило $3\sigma$

Мы уже разобрали, что данные реального мира ведут себя по-разному:
1. Одни признаки складываются из сотен случайных факторов и образуют красивый симметричный колокол Гаусса (как `bmi` благодаря ЦПТ).
2. Другие имеют жесткую границу снизу и единичные катастрофические выбросы, образуя тяжелый правый хвост (как страховые расходы `charges`).

Но как выразить эти явления в коде одной строчкой и числом, а не просто всматриваться в графики гистограмм? Для этого математическая статистика предлагает специальные числовые критерии.

### 1. Мера скошенности: Коэффициент асимметрии Пирсона

Мы уже знаем:
- **Выборочное среднее ($\mu$):** стремится минимизировать сумму квадратов ошибок ($L_2$-оптимум), поэтому единичный гигантский выброс утягивает его за собой.
- **Медиана ($\text{Med}$):** делит выборку ровно пополам ($L_1$-робастный оптимум) и практически не реагирует на выбросы.

Если распределение идеально симметрично (как у кубиков или нормального колокола), то **$\mu \approx \text{Med}$**. 
Если же в выборке есть длинный «хвост» супер-дорогих чеков или огромных зарплат, **среднее улетает вправо, а медиана остается на месте ($\mu > \text{Med}$)**.

Карл Пирсон предложил оцифровать эту разницу в виде **второго коэффициента асимметрии (Skewness)**:

$$\text{Skew} = \frac{3(\mu - \text{Med})}{\sigma}$$

Если измерять расходы в рублях, разница $(\mu - \text{Med})$ будет в тысячах, а если в миллионах — в сотых долях. Деление на стандартное отклонение $\sigma$ нормирует масштаб, позволяя сравнивать асимметрию признаков с абсолютно разными единицами измерения (например, возраст в годах и расходы в долларах).

#### Как интерпретировать значение:
- **$\text{Skew} \approx 0$ (от $-0.5$ до $0.5$):** распределение практически симметрично (среднее совпадает с медианой). Модель линейной регрессии чувствует себя комфортно.
- **$\text{Skew} > 1$ (сильный правый хвост):** в выборке присутствуют аномально большие значения. MSE-лосс будет страдать и завышать прогноз для большинства типичных наблюдений.
- **$\text{Skew} < -1$ (сильный левый хвост):** хвост из редких аномально низких значений.

### 2. Экспресс-тест на нормальность и поиск аномалий: Правило трёх сигм ($3\sigma$)

Если случайная величина подчиняется **нормальному закону** $X \sim \mathcal{N}(\mu, \sigma^2)$, то её разброс строго математически предсказуем: вероятность отклониться от центра $\mu$ на определенное число стандартных отклонений $\sigma$ падает по экспоненте.

Интегрируя плотность нормального распределения, математики получили **Правило трёх сигм (эмпирическое правило)**:
- В диапазон $[\mu - 1\sigma, \mu + 1\sigma]$ гарантированно попадает **$\approx 68.27\%$** всех наблюдений.
- В диапазон $[\mu - 2\sigma, \mu + 2\sigma]$ попадает **$\approx 95.45\%$** всех наблюдений.
- В диапазон $[\mu - 3\sigma, \mu + 3\sigma]$ попадает **$\approx 99.73\%$** всех наблюдений.

Наблюдения, лежащие дальше трёх сигм от центра ($\vert{}x - \mu\vert{} > 3\sigma$), встречаются с вероятностью меньше **$0.27\%$** (менее 3 случаев на 1000 наблюдений).

#### Зачем это нужно?
1. **Автоматический детектор выбросов (Outlier Detection):**
   Если при предварительной очистке данных вы видите точку за пределами $[\mu - 3\sigma, \mu + 3\sigma]$, это почти наверняка экстремальная аномалия, ошибка ввода или тот самый катастрофический случай, который утянет веса модели на себя.
2. **Проверка гипотезы о нормальности признака:**
   Если вы считаете долю точек внутри $[\mu \pm 3\sigma]$ и получаете, например, $91\%$ вместо $99.7\%$, значит, распределение **не нормальное** (у него тяжелые хвосты), и к нему нельзя слепо применять методы, рассчитанные на гауссианы.

In [ ]:
# 1. Сгенерируем нормальный колокол
np.random.seed(42)
normal_feature = np.random.normal(loc=30.0, scale=6.0, size=10000)

# 2. Создадим скошенное распределение с редкими выбросами
skewed_feature = np.concatenate([
    np.random.exponential(scale=5000, size=9500),  # масса типичных обращений
    np.random.uniform(low=40000, high=60000, size=500) # редкие случаи
])

def demo_stats(name, arr):
    mu = np.mean(arr)
    med = np.median(arr)
    sigma = np.std(arr)
    skew = 3 * (mu - med) / sigma
    
    # Считаем процент точек внутри 3-х сигм
    in_3sigma = np.mean((arr >= mu - 3*sigma) & (arr <= mu + 3*sigma)) * 100
    
    print(name)
    print(f"Среднее (mu): {mu:.2f} | Медиана (Med): {med:.2f}")
    print(f"Коэффициент асимметрии Пирсона: {skew:.3f}")
    print(f"Попадает в [mu +/- 3*sigma]: {in_3sigma:.2f}% (теория: 99.73%)\n")

demo_stats("Нормальный признак (колокол)", normal_feature)
demo_stats("Скошенный признак с правым хвостом", skewed_feature)

#### Задание 2.1. Ручной расчет среднего и медианы

Реализуйте функцию `my_mean_median(x)`:
- Принимает 1D-массив `x` размерности $(N,)$.
- Возвращает кортеж `(mean_val, median_val)`.
- Медиана для четного числа элементов считается как среднее двух центральных значений.

*Ограничение: запрещено использовать `np.mean`, `np.median`, а также методы `.mean()`.*
*Разрешено использовать: `np.sum()`, `len()`, `np.sort()`.*

In [ ]:
def my_mean_median(x: np.ndarray) -> tuple[float, float]:
    """Вычисляет среднее и медиану одномерного массива x без встроенных функций np.mean и np.median."""
    # YOUR CODE HERE
    pass

#### Задание 2.2. Ручной расчет дисперсии и стандартного отклонения

Формула выборочной дисперсии:
$$Var(x) = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2, \quad \sigma = \sqrt{Var(x)}$$

Реализуйте функцию `my_var_std(x)`:
- Принимает 1D-массив `x` размерности $(N,)$.
- Возвращает кортеж `(var_val, std_val)`.

*Ограничение: запрещено использовать `np.var`, `np.std`, а также методы `.var()` и `.std()`.*

In [ ]:
def my_var_std(x: np.ndarray) -> tuple[float, float]:
    """Вычисляет дисперсию и стандартное отклонение одномерного массива x."""
    # YOUR CODE HERE
    pass

#### Задание 2.3. Коэффициент асимметрии Пирсона

Реализуйте функцию `calculate_skewness(x)`:
- Принимает 1D-массив `x`.
- Считает асимметрию по формуле:
$$\text{Skew} = \frac{3(\mu - \text{Med})}{\sigma}$$
- Возвращает число `float`.

*Подсказка: используйте функции `my_mean_median` и `my_var_std`, написанные ранее.*

In [ ]:
def calculate_skewness(x: np.ndarray) -> float:
    """Вычисляет коэффициент асимметрии Пирсона: 3 * (mean - median) / std."""
    # YOUR CODE HERE
    pass

#### Задание 2.4. Проверка правила трех сигм

Реализуйте функцию `check_three_sigma(x)`:
- Принимает 1D-массив `x`.
- Рассчитывает среднее $\mu$ и стандартное отклонение $\sigma$.
- Находит **доли** точек (числа от `0.0` до `1.0`), попавших в интервалы:
  1. $[\mu - \sigma, \mu + \sigma]$
  2. $[\mu - 2\sigma, \mu + 2\sigma]$
  3. $[\mu - 3\sigma, \mu + 3\sigma]$
- Возвращает кортеж из трех долей `(p1, p2, p3)`.

*Ограничение: используйте булевы маски NumPy (без циклов `for`).*

Чтобы проверить условие для всех элементов массива сразу (без циклов `for`), в NumPy используются операторы сравнения:
- `arr > 10` вернет массив из `True` и `False`.
- Чтобы объединить несколько условий, используются побитовые операторы:
  - `&` — логическое **И**
  - `|` — логическое **ИЛИ**
  - **Каждое условие обязательно берется в круглые скобки!**

Поскольку Python воспринимает `True` как `1`, а `False` как `0`:
- `np.sum(маска)` — посчитает **количество** подходящих элементов.
- `np.mean(маска)` — сразу вернет **долю (процент от 0 до 1)** подходящих элементов.

In [ ]:
def check_three_sigma(x: np.ndarray) -> tuple[float, float, float]:
    """Возвращает доли наблюдений, попадающих в интервалы 1, 2 и 3 сигм от среднего."""
    # YOUR CODE HERE
    pass

Проверьте результаты с помощью тестов

In [ ]:
from checker_raw import (
    test_my_mean_median,
    test_my_var_std,
    test_calculate_skewness,
    test_check_three_sigma
)

test_my_mean_median(my_mean_median)
test_my_var_std(my_var_std)
test_calculate_skewness(calculate_skewness)
test_check_three_sigma(check_three_sigma)

## Блок 3. Базовый Pandas: форма, срезы и перевод в NumPy

Для подготовки таблицы Pandas к математической модели нам нужны три базовых действия:
1. **Проверка формы (`.shape`):** узнать число объектов $N$ (строк) и признаков (столбцов).
2. **Срезы (`.iloc` и выбор столбцов):** быстрый доступ к нужным строкам и подмножествам признаков.
3. **Разделение и конвертация в NumPy (`.to_numpy()`):** получение матрицы объектов-признаков $X \in \mathbb{R}^{N \times d}$ и вектора таргета $y \in \mathbb{R}^N$.

In [ ]:
import pandas as pd

# Пример таблицы данных
demo_df = pd.DataFrame({
    'age': [25, 30, 45, 50],
    'bmi': [22.4, 28.1, 31.0, 26.5],
    'children': [0, 2, 1, 3],
    'charges': [3200.0, 4800.0, 9500.0, 11200.0]  # таргет
})

# 1. Размерность таблицы (кортеж: строки, столбцы)
print("1. Форма таблицы (N, total_cols):", demo_df.shape)

# 2. Срезы через .iloc (строки, столбцы по индексам)
# Первые 2 строки и первые 2 столбца
print("\n2. Срез .iloc[:2, :2]:\n", demo_df.iloc[:2, :2])

# 3. Разделение на матрицу признаков X и вектор таргета y с переводом в NumPy:
X_demo = demo_df.drop(columns=['charges']).to_numpy()
y_demo = demo_df['charges'].to_numpy()

print("\n3. Форма матрицы признаков X:", X_demo.shape)  # 2D (4, 3)
print("   Форма вектора таргета y:", y_demo.shape) # 1D (4,)

#### Задание 3.1. Определение размерности выборки

Реализуйте функцию `get_dataset_dimensions(df)`:
- Принимает `pd.DataFrame`.
- Возвращает кортеж `(n_rows, n_cols)`:
  - `n_rows` (int) — количество объектов (строк).
  - `n_cols` (int) — количество колонок (признаков вместе с таргетом).

In [ ]:
def get_dataset_dimensions(df: pd.DataFrame) -> tuple[int, int]:
    """Возвращает количество строк и столбцов датафрейма."""
    # YOUR CODE HERE
    pass

#### Задание 3.2. Срезы строк и столбцов

Реализуйте функцию `slice_dataframe(df, n_rows, col_names)`:
- Принимает датафрейм `df`, число строк `n_rows` (int) и список названий колонок `col_names` (list of str).
- Возвращает новый `pd.DataFrame`, содержащий **первые `n_rows` строк** только для **выбранных столбцов `col_names`**.

In [ ]:
def slice_dataframe(df: pd.DataFrame, n_rows: int, col_names: list[str]) -> pd.DataFrame:
    """Возвращает срез датафрейма: первые n_rows строк для заданных столбцов."""
    # YOUR CODE HERE
    pass

#### Задание 3.3. Разделение матрицы признаков и вектора таргета в NumPy

Реализуйте функцию `prepare_features_and_target(df, target_col)`:
- Принимает датафрейм `df` и строковое имя целевой колонки `target_col`.
- Извлекает `target_col` в виде **одномерного массива NumPy** формы $(N,)$ типа `float64`.
- Формирует матрицу признаков $X$ (все столбцы, кроме `target_col`)[cite: 3, 30] в виде **двумерного массива NumPy** формы $(N, d)$ типа `float64`.
- Возвращает кортеж `(X, y)`. Исходный датафрейм `df` изменяться не должен.

In [ ]:
def prepare_features_and_target(df: pd.DataFrame, target_col: str) -> tuple[np.ndarray, np.ndarray]:
    """Разделяет датафрейм на матрицу признаков X и вектор таргета y в формате NumPy."""
    # YOUR CODE HERE
    pass

Проверьте результаты с помощью тестов

In [ ]:
from checker import (
    test_get_dataset_dimensions,
    test_slice_dataframe,
    test_prepare_features_and_target
)

test_get_dataset_dimensions(get_dataset_dimensions)
test_slice_dataframe(slice_dataframe)
test_prepare_features_and_target(prepare_features_and_target)

## Блок 4. Оптимизация: Градиентный спуск, обучение моделей и сравнение

Обучение линейной модели — это задача поиска весов $w^*$, при которых функция ошибки минимальна:
$$w^* = \arg\min_w \mathcal{L}(w)$$

Чтобы спуститься к оптимуму ошибки, мы используем **антиградиент** — направление наискорейшего убывания функции:
$$w^{(t+1)} = w^{(t)} - \eta \nabla \mathcal{L}(w^{(t)})$$
где $\eta$ (learning rate) — шаг обучения.

### Градиенты функций потерь

Обозначим вектор ошибок прогноза: $e = \hat{y} - y = Xw - y \in \mathbb{R}^{N \times 1}$.

#### 1. Mean Squared Error (MSE, $L_2$-лосс)
$$\mathcal{L}_{MSE}(w) = \frac{1}{N} \sum_{i=1}^N (X_i w - y_i)^2 = \frac{1}{N} \Vert{}Xw - y\Vert{}_2^2$$

Производная от квадрата $(u^2)' = 2u$. Применяя правило сложной функции, получаем компактный матричный градиент:
$$\nabla \mathcal{L}_{MSE}(w) = \frac{2}{N} X^T (Xw - y)$$

#### 2. Mean Absolute Error (MAE, $L_1$-лосс)
$$\mathcal{L}_{MAE}(w) = \frac{1}{N} \sum_{i=1}^N \vert{}X_i w - y_i\vert{}$$

Производная модуля: $(\vert{}u\vert{})' = \text{sign}(u)$ (знак ошибки: $+1$, если переоценили, $-1$, если недооценили, и $0$, если угадали точно).
Матричная формула градиента для MAE выглядит аналогично, но вместо самих ошибок берется их знак:
$$\nabla \mathcal{L}_{MAE}(w) = \frac{1}{N} X^T \text{sign}(Xw - y)$$
*(В NumPy для знака используется функция `np.sign`)*.

#### Задание 4.1. Градиент функции потерь MSE

Реализуйте функцию `compute_mse_gradient(X, y, w)`:
- Принимает:
  - `X`: матрица признаков формы $(N, d)$.
  - `y`: вектор таргета формы $(N,)$.
  - `w`: вектор весов формы $(d,)$.
- Вычисляет вектор градиента по формуле:
  $$\nabla \mathcal{L}_{MSE}(w) = \frac{2}{N} X^T (Xw - y)$$
- Возвращает одномерный массив формы $(d,)$.

In [ ]:
def compute_mse_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Вычисляет градиент MSE по весам w."""
    # YOUR CODE HERE
    pass

#### Задание 4.2. Градиент функции потерь MAE

Реализуйте функцию `compute_mae_gradient(X, y, w)`:
- Принимает: `X` формы $(N, d)$, `y` формы $(N,)$, `w` формы $(d,)$.
- Вычисляет вектор градиента по формуле:
  $$\nabla \mathcal{L}_{MAE}(w) = \frac{1}{N} X^T \text{sign}(Xw - y)$$
- Возвращает одномерный массив формы $(d,)$.

*Подсказка: используйте `np.sign(X @ w - y)`.*

In [ ]:
def compute_mae_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Вычисляет градиент MAE по весам w."""
    # YOUR CODE HERE
    pass

#### Задание 4.3. Цикл обучения градиентным спуском

Реализуйте функцию `fit_linear_regression(X, y, loss_type='mse', lr=0.05, n_steps=100)`:
1. Инициализирует веса нулями: `w = np.zeros(X.shape[1])`.
2. В цикле на `n_steps` итераций:
   - Делает прогноз: $\hat{y} = Xw$.
   - Считает значение текущей ошибки ($MSE$ или $MAE$ в зависимости от `loss_type`) и добавляет его в список `loss_history`.
   - Считает градиент с помощью функции `compute_mse_gradient` или `compute_mae_gradient`.
   - Обновляет веса по формуле антиградиента: $w = w - \eta \cdot \nabla \mathcal{L}(w)$.
3. Возвращает кортеж `(w, loss_history)`:
   - `w` — итоговый вектор весов.
   - `loss_history` — список значений функции потерь на каждом шаге.

In [ ]:
def fit_linear_regression(
    X: np.ndarray, 
    y: np.ndarray, 
    loss_type: str = 'mse', 
    lr: float = 0.05, 
    n_steps: int = 100
) -> tuple[np.ndarray, list[float]]:
    """Обучает линейную регрессию методом градиентного спуска (поддерживает loss_type 'mse' и 'mae')."""
    # YOUR CODE HERE
    pass

Проверьте результаты с помощью тестов

In [ ]:
from checker import (
    test_compute_mse_gradient,
    test_compute_mae_gradient,
    test_fit_linear_regression
)

test_compute_mse_gradient(compute_mse_gradient)
test_compute_mae_gradient(compute_mae_gradient)
test_fit_linear_regression(fit_linear_regression)

## Блок 5. Практическое применение: собираем собственный ML-пайплайн

Теперь применим наши функции к реальным данным.

Мы будем работать с датасетом автомобилей **`cars.csv`**. Наша цель — предсказать топливную экономичность автомобиля (**`mpg`** — миль на галлон: чем больше значение, тем меньше расход топлива) по его техническим характеристикам.

### Описание признаков датасета:
- `mpg` — **целевая переменная**: топливная экономичность (миль на галлон, `float`).
- `cylinders` — количество цилиндров в двигателе (`int`).
- `displacement` — рабочий объем двигателя в кубических дюймах (`float`).
- `horsepower` — мощность двигателя в лошадиных силах (`float`).
- `weight` — масса автомобиля в фунтах (`float`).
- `acceleration` — время разгона от 0 до 60 миль/ч в секундах (`float`).
- `model year` (или `model_year`) — год выпуска модели (например, `70` = 1970 год, `int`).

#### Задание 5.1. Загрузка данных и первичный осмотр (применяем Блок 3)

Загрузим таблицу `cars.csv` и проверим её размерность и первые строки с помощью написанных вами функций `get_dataset_dimensions` и `slice_dataframe`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Загружаем подготовленный датасет
df_cars = pd.read_csv('cars.csv')

# TODO: Вызовите вашу функцию get_dataset_dimensions, чтобы узнать число строк и столбцов
n_rows, n_cols = ...  # YOUR CODE HERE

print(f"Размер датасета: {n_rows} автомобилей и {n_cols} колонок")

# TODO: Вызовите вашу функцию slice_dataframe, чтобы вывести первые 5 строк 
# для колонок ['mpg', 'horsepower', 'weight', 'acceleration']
preview_df = ...  # YOUR CODE HERE
preview_df

#### Задание 5.2. Исследование распределений

Проверим, как ведут себя время разгона (`acceleration`), мощность (`horsepower`) и наш таргет (`mpg`), используя функции из **Блока 2**

In [ ]:
features_to_check = ['acceleration', 'horsepower', 'mpg']

for col in features_to_check:
    arr = df_cars[col].to_numpy()
    
    # TODO: Вызовите ваши функции из Блока 2 для массива arr
    mean_val, median_val = ...  # my_mean_median
    var_val, std_val = ...      # my_var_std
    skew_val = ...              # calculate_skewness
    p1, p2, p3 = ...            # check_three_sigma
    
    print(f"Признак: {col}")
    print(f"Среднее (L2): {mean_val:.2f} | Медиана (L1): {median_val:.2f} | Std (sigma): {std_val:.2f}")
    print(f"Коэффициент асимметрии Пирсона: {skew_val:.3f}")
    print(f"Правило 3 сигм: 1σ = {p1*100:.1f}%, 2σ = {p2*100:.1f}%, 3σ = {p3*100:.1f}% (теория Гаусса: 68.3%, 95.5%, 99.7%)\n")

In [ ]:
# Визуализируем распределения, чтобы сопоставить графики с полученными числами
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes, ['acceleration', 'horsepower', 'mpg'], ['mediumseagreen', 'coral', 'cornflowerblue']):
    sns.histplot(df_cars[col], kde=True, ax=ax, color=color)
    ax.axvline(df_cars[col].mean(), color='red', linestyle='--', lw=2, label=f"Mean: {df_cars[col].mean():.1f}")
    ax.axvline(df_cars[col].median(), color='navy', linestyle='-', lw=2, label=f"Median: {df_cars[col].median():.1f}")
    ax.set_title(f"Распределение {col}")
    ax.legend()

plt.tight_layout()
plt.show()

#### Задание 5.3. Препроцессинг: переход к матрицам и стандартизация (применяем Блоки 1 и 3)

Перед запуском градиентного спуска подготовим данные:
1. Отделим матрицу признаков $X$ от вектора целевой переменной $y$ (`mpg`) с помощью `prepare_features_and_target`.
2. Проведем **Z-score стандартизацию** признаков и таргета ($\tilde{x}_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$), чтобы уравнять масштаб величин (вес измеряется в тысячах фунтов, а цилиндры — в единицах).
3. Добавим слева столбец из единиц ($x_0 \equiv 1$) с помощью `add_bias_term`.

In [ ]:
# 1. TODO: Разделите df_cars на матрицу X_raw и вектор y_raw (таргет 'mpg') вашей функцией из Блока 3
X_raw, y_raw = ...  # prepare_features_and_target

# 2. TODO: Стандартизируйте матрицу признаков X_raw, используя ваши функции center_matrix и compute_column_stats
X_means, X_stds = ...  # compute_column_stats(X_raw)
X_centered = ...       # center_matrix(X_raw)
X_scaled = X_centered / X_stds

# 3. TODO: Стандартизируйте вектор таргета y_raw, используя my_mean_median и my_var_std
y_mean, _ = ...        # my_mean_median(y_raw)
_, y_std = ...         # my_var_std(y_raw)
y_scaled = (y_raw - y_mean) / y_std

# 4. TODO: Добавьте столбец единиц (bias) к X_scaled вашей функцией add_bias_term
X_mat = ...            # add_bias_term(X_scaled)

print("Форма итоговой матрицы X_mat:", X_mat.shape)
print("Форма вектора y_scaled:", y_scaled.shape)

#### Задание 5.4. Влияние шага обучения ($\eta$) и обучение моделей MSE vs MAE (применяем Блок 4)

На лекции мы видели, как выбор скорости обучения $\eta$ (Learning Rate) влияет на сходимость:
- Слишком малый $\eta$ — оптимизатор ползет к минимуму слишком медленно.
- Оптимальный $\eta$ — быстрая и плавная сходимость на дно чаши потерь.

Проверим это на нашей функции `fit_linear_regression`, а затем обучим две финальные модели: одну с минимизацией **MSE ($L_2$)**, а вторую — с минимизацией **MAE ($L_1$)**.

In [ ]:
lrs = [0.005, 0.05, 0.25]
plt.figure(figsize=(8, 4))

for lr in lrs:
    _, losses = fit_linear_regression(X_mat, y_scaled, loss_type='mse', lr=lr, n_steps=80)
    plt.plot(losses, lw=2, label=f"lr = {lr}")

plt.title("Влияние скорости обучения (lr) на сходимость градиентного спуска")
plt.xlabel("Шаг итерации")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, linestyle=':')
plt.show()

# TODO: Обучите две финальные модели (MSE и MAE) при lr=0.1 и n_steps=150
w_mse, losses_mse = ...  # fit_linear_regression для 'mse'
w_mae, losses_mae = ...  # fit_linear_regression для 'mae'

### Шаг 5.5. Сравнение качества моделей и интерпретация весов

Теперь получим предсказания обеих моделей с помощью вашей функции `predict_linear`, вернем их из стандартизированного масштаба в реальные мили на галлон (`mpg`) и сравним итоговые метрики **MAE**, **MSE** и **RMSE**.

In [ ]:
# 1. TODO: Получите предсказания моделей в стандартизированном масштабе через вашу функцию predict_linear
y_pred_scaled_mse = ...  # predict_linear(X_mat, w_mse)
y_pred_scaled_mae = ...  # predict_linear(X_mat, w_mae)

# Переводим предсказания обратно в реальные единицы (мили на галлон)
y_pred_mse = y_pred_scaled_mse * y_std + y_mean
y_pred_mae = y_pred_scaled_mae * y_std + y_mean

# 2. Считаем метрики качества на реальном масштабе
def get_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    return mae, mse, rmse

mae_1, mse_1, rmse_1 = get_metrics(y_raw, y_pred_mse)
mae_2, mse_2, rmse_2 = get_metrics(y_raw, y_pred_mae)

metrics_df = pd.DataFrame({
    'Метрика': ['MAE (L1, миль/галлон)', 'MSE (L2, кв. ед.)', 'RMSE (миль/галлон)'],
    'Модель MSE': [round(mae_1, 3), round(mse_1, 3), round(rmse_1, 3)],
    'Модель MAE': [round(mae_2, 3), round(mse_2, 3), round(rmse_2, 3)]
})
print("Сравнение метрик")
print(metrics_df.to_string(index=False))

# 3. Сравнение выученных весов
feature_names = ['bias'] + [col for col in df_cars.columns if col != 'mpg']
weights_comparison = pd.DataFrame({
    'Признак': feature_names,
    'Вес (MSE модель)': np.round(w_mse, 4),
    'Вес (MAE модель)': np.round(w_mae, 4)
}).sort_values(by='Вес (MSE модель)', key=abs, ascending=False)

print("\nВеса моделей")
print(weights_comparison.to_string(index=False))

# Визуализация весов
weights_melted = weights_comparison[weights_comparison['Признак'] != 'bias'].melt(
    id_vars='Признак', var_name='Модель', value_name='Вес'
)
plt.figure(figsize=(9, 4.5))
sns.barplot(data=weights_melted, x='Вес', y='Признак', hue='Модель', palette=['crimson', 'royalblue'])
plt.axvline(0, color='black', linestyle='--', alpha=0.7)
plt.title("Сравнение весов признаков: MSE vs MAE")
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()